# Interactive Visualization & Location Validation Demo

This notebook demonstrates:
1. **Location Validation**: Verify that generated bounds match the intended location
2. **Interactive Plotly Visualizations**: Explore UE mobility with interactive controls

## Setup

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

# Add maveric root to path
maveric_root = Path.cwd().parent.parent.parent.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Maveric root: {maveric_root}")
print(f"Current directory: {Path.cwd()}")

In [ ]:
# Import visualization functions
from radp.digital_twin.agentic_mobility.visualization import (
    validate_location_bounds,
    plot_bounds_on_map,
    plot_ue_wise_interactive,
    plot_tick_wise_interactive,
)

print("✓ Successfully imported visualization functions!")

## Load Generated Data

In [ ]:
# Load CSV and metadata
data_dir = Path.cwd() / "generated_ues"
csv_files = list(data_dir.glob("*.csv"))

print('available csv files: ' + str(csv_files))

if not csv_files:
    print("⚠️  No data files found!")
    print("Please run end_to_end_example.py first to generate data.")
else:
    # Load first CSV
    selected_csv = csv_files[-1]
    df = pd.read_csv(selected_csv)
    
    # Load corresponding metadata
    metadata_file = selected_csv.with_name(selected_csv.stem + "_metadata.json")
    if metadata_file.exists():
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
    else:
        metadata = None
    
    print(f"✓ Loaded: {selected_csv.name}")
    print(f"  - UEs: {df['mock_ue_id'].nunique()}")
    print(f"  - Ticks: {df['tick'].nunique()}")
    print(f"  - Total positions: {len(df)}")
    print(f"\nDataFrame preview:")
    display(df.head())

# Part 1: Location Validation

## Validate that spatial bounds match the intended location

This performs 5 reverse geocoding calls:
- 1 for the center point
- 4 for each corner (NW, NE, SW, SE)

Then validates that all points match the query intent location.

In [ ]:
if metadata and 'spatial_bounds' in metadata:
    print("Performing location validation...")
    print(f"Query location: {metadata.get('query_intent', {}).get('location')}")
    print("\n" + "="*70)
    
    # Run validation
    validation_result = validate_location_bounds(metadata, threshold=0.7)
    
    print("\nValidation Results:")
    print("="*70)
    
    if validation_result['is_match']:
        print(f"✓ LOCATION VALIDATED")
    else:
        print(f"✗ LOCATION MISMATCH")
    
    print(f"\nOverall Confidence: {validation_result['overall_confidence']:.1%}")
    print(f"Consistency Score: {validation_result['consistency_score']:.1%}")
    print(f"Query Location: {validation_result['query_location']}")
    
    print("\nDetected Locations (5 points):")
    print("-"*70)
    for point_name, location in validation_result['detected_locations'].items():
        confidence = validation_result['point_confidences'][point_name]
        city = location.get('city', 'Unknown')
        country = location.get('country', 'Unknown')
        print(f"  {point_name.upper():8s}: {city}, {country} (confidence: {confidence:.1%})")
    
    if validation_result['warnings']:
        print("\n⚠️  Warnings:")
        for warning in validation_result['warnings']:
            print(f"  - {warning}")
    
    print("\n" + "="*70)
else:
    print("⚠️  No metadata or spatial_bounds available for validation")

## Visualize Bounds on World Map

Show the spatial bounds as a rectangle on an interactive world map with all 5 validation points.

In [ ]:
if metadata and 'spatial_bounds' in metadata and validation_result:
    print("Creating world map visualization...")
    
    fig_map = plot_bounds_on_map(
        validation_result,
        title="Spatial Bounds Validation - World Map View",
        figsize=(1200, 800)
    )
    
    fig_map.show()
    
    print("\n💡 Interactive Map Features:")
    print("   - Red rectangle shows the spatial bounds")
    print("   - Blue circle = Center point")
    print("   - Green diamonds = Corner points (NW, NE, SW, SE)")
    print("   - Hover over points to see full address details")
    print("   - Legend shows all 5 points with their geocoded addresses")
else:
    print("⚠️  No validation result available for map visualization")

# Part 2: Interactive Plotly Visualizations

## Mode A: UE-wise Interactive View

Select a UE ID from the dropdown → see the full track with start/end markers.

In [ ]:
if csv_files:
    print("Creating UE-wise interactive visualization...")
    
    # Plot first 10 UEs for performance
    # num_ues_to_plot = min(10, df['mock_ue_id'].nunique())
    ue_ids_subset = sorted(df['mock_ue_id'].unique())#[:num_ues_to_plot]
    
    fig_ue_wise = plot_ue_wise_interactive(
        df,
        ue_ids=ue_ids_subset,
        show_arrows=True,
        figsize=(1000, 700)
    )
    
    fig_ue_wise.show()
    
    # print(f"\n💡 Use the dropdown menu to switch between {num_ues_to_plot} UEs")
    print("   - Green circle = Start point")
    print("   - Red square = End point")
    print("   - Hover over points to see details")

## Mode B: Tick-wise Interactive View

Use the slider to select a tick → see all UE positions at that moment.
Press PLAY to animate through time!

In [ ]:
if csv_files:
    print("Creating tick-wise interactive visualization...")
    
    fig_tick_wise = plot_tick_wise_interactive(
        df,
        initial_tick=0,
        color_by_ue=True,
        show_trails=False
    )
    
    fig_tick_wise.show()
    
    print("\n💡 Controls:")
    print("   - Click PLAY button to animate")
    print("   - Use slider to select specific tick")
    print("   - Colors represent different UEs")

## Saving Interactive Plots to HTML

You can save any of these figures as standalone HTML files:

In [ ]:
# Example: Save UE-wise plot to HTML
if csv_files:
    output_path = Path.cwd() / "interactive_ue_tracks.html"
    fig_ue_wise.write_html(str(output_path))
    print(f"✓ Saved interactive plot to: {output_path}")
    print("  Open this file in a web browser to view the interactive plot!")